In [ ]:
def readInteractions(f_name):
	try:
		f_handle = open(f_name,"r")
	except IOError as e:
		print("I/O error: can't open interactions file.")
		sys.exit()


	interactions = []
	int_file = open(f_name,"r")
	for line in int_file:
		interactions.append(line.strip().split(sep=","))
		interactions[-1][3] = (float(interactions[-1][3]))**2
		interactions[-1][4] = (float(interactions[-1][4]))**2
		if(interactions[-1][1] > interactions[-1][2]):
			temp = interactions[-1][1]
			interactions[-1][1] = interactions[-1][2]
			interactions[-1][2] = temp

	f_handle.close()
	int_list = sorted(set([i[0] for i in interactions]))

	return interactions, int_list

In [ ]:
f_name = 'interactions.csv'
result = readInteractions(f_name)
interactions = result[0]
int_list = result[1]

print(interactions)
print('br')
print(int_list)

In [ ]:
def __str__interactions(interactions):
	temp = ""
	x = 0
	for i in interactions:
		#print(i[0][4]) 
		#print(x)
		temp += "%-12s| %s-%s | %5.2f -%6.2f\n" %(i[0],i[1],i[2],i[3],i[4]) 
		x=x+1
	return temp

In [ ]:
def printInteractions(interactions):
	print(__str__interactions(interactions))

In [ ]:
printInteractions(interactions)

In [ ]:
def readAtom_Types(f_name):
	try:
		f_handle = open(f_name,"r")
	except IOError as e:
		print("I/O error: can't open atom types file.")
		sys.exit()

        
	types = set() # initialize with an empty set
	atm_data = dict() # initialize with an empty dictionary

	temp = [] # declearing an empty array
	res_name = ""
	for line in f_handle:
		temp = line.strip('\n').split(",") 
		if res_name != temp[0]:
			atm_data[temp[0]] = dict() 
			res_name = temp[0]
		atm_data[res_name][temp[1]] = [ i for i in temp[2:]]
		types |= set(temp[2:])
	f_handle.close()

	types = sorted(types)
	return atm_data,types

In [ ]:
f_name = "atom_types.csv";
result = readAtom_Types(f_name)

atm_data = result[0]
types = result[1]

a_types = atm_data

a_type_list = types

print(a_types)
print('br')
print(a_type_list)

In [ ]:
def __str__Atom_Types(atm_data):

	temp = ""
	for k, v in sorted(atm_data.items()):
		temp += "\n" + k.__str__()
		#print(temp)
		for i, j in sorted(v.items()):
			temp += "\n %6s |" % (i.__str__())
			for typ in sorted(j):
				temp += "%4s" % typ
	
	return temp

In [ ]:
def printAtom_Types(atm_data):
	print(__str__Atom_Types(atm_data))

In [ ]:
print(printAtom_Types(atm_data))

In [ ]:
class TypeCode:
	def __init__(self,atom_types,int_types):
		
		type_map = enumerate(sorted(atom_types) + sorted(int_types))
		#print(list(type_map))
		self.type_map = {v:(1<<k) for k,v in type_map}
		#print(self.type_map)
		self.keys = sorted(self.type_map.values())
		self.rev = {k:v for v,k in self.type_map.items()}
		print(self.type_map)

	def __getitem__(self,t):
		return self.type_map[t]


	def get_rev(self,n):
		return self.rev[n]

	def type(self,n):
		return [self.rev[i] for i in filter(lambda x: x & n,self.keys)]
	
	def code(self,n):
		return [i for i in filter(lambda x: x & n,self.keys)]

	def len(self):
		return len(self.type_map)

	def max(self):
		return 1<<(self.len()-1)

	def __str__(self):
		return self.type_map.__str__()
		

In [ ]:
typeCode = TypeCode(a_type_list,int_list)

In [ ]:
import json


class TypeMap:
	def __init__(self,json_file):
		with open(json_file,"r") as json_file_holder:
			self.type_map = json.loads(json_file_holder.read())

	def __str__(self):
		return self.type_map.__str__()

	def __getitem__(self,t):
		return self.type_map[t]

In [ ]:
f_name = "typenames.json"
typenames = TypeMap(f_name)
print (typenames)

In [ ]:
	import os
	import sys
	from pathlib import Path 
	#GSPANPATH = os.getcwd() + "/result"
	GSPANPATH = os.getcwd() + "/gSpan/gSpan-64"
	pdbids_file = "pdb_chain.txt"
	path = Path(GSPANPATH)


In [ ]:
	import logging
	path.mkdir(parents=True,exist_ok=True)
	path = str(path)
		
	logging.basicConfig(level=logging.DEBUG)
	

    

In [ ]:
"""Xdef read_pdbid_file(filename):
	chains = dict()
	with open(filename,"r") as input_pdbs:
		value = input_pdbs.read().split("\n")
		chains = list(map(lambda x: x.split(","),value))
		chains = list(filter(lambda x: x[0] != "#",chains))
		#for c in chains:
			#print(c)
			#print("--") 
			#v = len(c)//2
			#print(v)
			#print("----")           
			#for n in range(v): 
				#print(n)
				#print("------")
				#chains = c[2*n+1]
				#print(chains)
		#chains = [[c[0]] + [(c[2*n+1]) for n in range(len(c)//2)] for c in chains]
		chains = [[c[0]] + [(c[2*n+1],c[2*n+2]) for n in range(len(c)//2)] for c in chains]
		chains = {k:v for k,v in [(i[0],i[1:]) for i in chains] }

	pdbids = sorted(chains.keys())
	#pdbids = filter(None, pdbids)

	return pdbids,chains"""

In [ ]:
def read_pdbid_file(filename):
    chains = {}

    with open(filename, "r") as input_pdbs:
        lines = [line.strip() for line in input_pdbs if line.strip()]
        parsed = [[item.strip() for item in line.split(",")] for line in lines]
        parsed = [row for row in parsed if row and row[0] != "#"]

        cleaned = []
        for row in parsed:
            if len(row) < 3:
                continue

            pdb_id = row[0].lower()
            chain_pairs = [(row[2*n+1], row[2*n+2]) for n in range(len(row)//2)]
            cleaned.append([pdb_id] + chain_pairs)

        chains = {row[0]: row[1:] for row in cleaned}

    pdbids = sorted(chains.keys())

    return pdbids, chains

In [ ]:
pdbids,chains = read_pdbid_file(pdbids_file)

In [ ]:
print (pdbids)

In [ ]:
print (chains)

In [ ]:
def read_PDB_files(pdbids,directory="",show_info=True):

	path = Path.cwd() / directory
	path.mkdir(parents=True,exist_ok=True)

	# with open(pdbids) as input_pdbs:
	# 	pdbids = list(map(lambda x: x.split(","),input_pdbs.read().split("\n")))
	# 	pdbids = list(filter(lambda x: x[0][0] != "#",pdbids))

	pdb_structures = dict()
	info = dict()

	for i in pdbids:

		if (path / ("pdb%s.ent" %i.lower())).exists():
			f_name = str(path / ("pdb%s.ent" %i.lower()))

		elif (path / ("%s.pdb" %i.lower())).exists():
			f_name = str(path / ("%s.pdb" %i.lower()))

		else:
			f_name = pdbl().retrieve_pdb_file(pdb_code=i,pdir=directory,file_format='pdb')

    
		#chains = [i[2*n+1:2*n+3] for n in range(len(i)//2)]
		structure = PDBParser(QUIET=True).get_structure(i,f_name)

		pdb_structures[i] = structure
		
		num_models = len([j for j in structure.get_models()])
		if num_models > 1:
			info[i] = num_models
		
		#exit()
	#chains = {k:v for k,v in }
	# if show_info:
	# 	print("%d pdbs have multiple models:"%len(info))
	# 	for k,v in sorted(info.items()):
	# 		print("%s-(%d)"%(k,v))
	# 	print("Total number of models: %s"%sum(info.values()))

	return pdb_structures

In [ ]:
import os
import Bio.PDB.PDBList as pdbl
from Bio.PDB import PDBParser, PDBIO
pdb_directory = r"F:\research\biopythonChainExtractionPDB\merge"

pdb_structures = read_PDB_files(pdbids,directory=pdb_directory, show_info=True)

In [ ]:
pip install Bio

In [ ]:
print (pdb_structures)

In [ ]:
def write_pdb_files(structures,chains,directory=""):

	path = Path(".") / directory
	path.mkdir(parents=True,exist_ok=True)
	#print(path)

	#print("write pdb files:")
	for pdbid,pdb_structure in structures.items():
		#print(pdbid)	
		# models = map(lambda x: x.get_id(),pdb_structure.get_models())
		io = PDBIO()
		io.set_structure(pdb_structure)
		# for models in models:
		for s,t in chains[pdbid]:
			#print(pair)
			fname = str(path/("%s.%s.pdb" %(pdbid,s)))
			io.save(fname,ChainSelect([s,t]))

In [ ]:
class ChainSelect:
	def __init__(self,chains,model=0):
		self.chains = list(chains)
		self.model = model       

	def accept_chain(self,chain):
		if chain.get_id() in self.chains:
			#print(chain,chain.get_id())
			return True
		return False

	def accept_model(self,model):
		if model.get_id() == self.model:
			#print("File contains multiple models")
			return True
		return False

	def accept_residue(self,residue):
		return True

	def accept_atom(self,atom):
		return True

In [ ]:
pdb_files = write_pdb_files(pdb_structures,chains,directory="pdbs")

In [ ]:
def calc_contacts(pdb_obj,chains,interactions,atom_type,id="",ignore_models=True):
	
	if(ignore_models): 
		logging.warning("Ignoring multiple models in calc_contacts!")
	
	contacts = dict()

	for model in pdb_obj.get_models():
		
		model_contacts = dict()
		
		for chain1,chain2 in chains:
			atom_contacts = []
			
			####### Chain1
			atom_list1 = list(model[chain1].get_atoms())
			
			####### Chain2
			atom_list2 = list(model[chain2].get_atoms())
			
			####### Neighborhood Search data structure
			tree = NeighborSearch(atom_list2)
			for s_atom in atom_list1:
				if s_atom.parent.id[0] != " " or s_atom.name == "OXT":
					continue

				s_types = atom_type[s_atom.parent.resname].get(s_atom.name,[])

				if not s_types:
					continue
				
				for d_atom in tree.search(s_atom.coord,6,level="A"):
					
					if d_atom.parent.id[0] != " " or d_atom.name == "OXT":
						continue
					
					d_types = atom_type[d_atom.parent.resname].get(d_atom.name,[])
					
					if not d_types:
						continue
		
					for i in s_types:
						for j in d_types:
							count = 0
							for interaction in interactions:
								if sorted([i,j]) == sorted(interaction[1:3]):
									d = dist3D(s_atom.coord,d_atom.coord)
									if interaction[3] <= d <= interaction[4]:
										s = s_atom.serial_number
										contact = [s_atom,d_atom,interaction[0],i,j,d**0.5,s_types,d_types]
										
										atom_contacts.append(tuple(contact))
			
		#/////////////////////////////////////////////////////////////////////////////////////////////////////
		###### Output
			dtype = [('S_ATOM',object),('D_ATOM',object),('INTERACTION','U12'),('I_TYPE','U3'),
					('J_TYPE','U3'),('DISTANCE',float),('S_TYPES',object),('D_TYPES',object)]
			model_contacts_temp = np.array(atom_contacts,dtype=dtype)
			model_contacts[(chain1,chain2)] = model_contacts_temp

		contacts[model.get_id()] = model_contacts

		if ignore_models:
			break

	return contacts

def run_contacts(structures,chains,interactions,atom_type):
	contacts = dict()
	for pdbid in sorted(structures.keys()):
		contacts[pdbid] = calc_contacts(structures[pdbid],chains[pdbid],interactions,atom_type)

	return contacts

def dist3D(a,b):
	return (a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2 

In [ ]:
from Bio.PDB import NeighborSearch
import sys, numpy as np
contacts = run_contacts(pdb_structures,chains,interactions,a_types)

In [ ]:
print (contacts)

In [ ]:
def gen_graphs(contacts,typeCode,gfilename="graphs.txt",path=""):

	graphs = []

	for pdbid, contacts in sorted(contacts.items()):
		graphs += gen_graph(contacts,pdbid,typeCode)

	for i in range(len(graphs)):
		graphs[i].graph["id"] = i

	graphs_string = ""
	graphs_string =  "".join(["".join(i for i in nx.generate_gml(g)) + "#" for g in graphs])

	#Write graphs file
	with (Path(path)/gfilename).open(mode="w") as gfile:
		gfile.write(graphs_string)

	#Generate node and edge label sets
	node_labels = set()
	edge_labels = set()
	for g in graphs:
		node_labels |= set(nx.get_node_attributes(g,'type').values())
		edge_labels |= set(nx.get_edge_attributes(g,'type').values())
		
	node_labels = sorted(node_labels)
	edge_labels = sorted(edge_labels)

	edge_labels = fill_label_set(edge_labels)
	node_labels = fill_label_set(node_labels)
	
	return graphs, node_labels, edge_labels

def gen_graph(contacts,pdbid,typeCode):
	
	temp = []
	
	graphs_string = ""
	g_id=0
	
	for model_idx,model in sorted(contacts.items()):
		for chains,group in sorted(model.items()):
		#/////////////////////////////////////////////////////////////////////////////////////////////////////
		###### Graph Generation
			
			g = nx.Graph()
			
			for contact in group:
				#Atom 1
				full_id = contact[0].get_full_id()
				s_data = [('model',full_id[1]),('chain',full_id[2]),("isLigand", False),('residueNumber',full_id[3][1]),
							('residueName',contact[0].get_parent().get_resname()),("atomName",full_id[4][0])]
				s_atom = full_id

				#Atom 2
				full_id = contact[1].get_full_id()
				d_data = [('model',full_id[1]),('chain',full_id[2]),("isLigand", True),('residueNumber',full_id[3][1]),
							('residueName',contact[1].get_parent().get_resname()),("atomName",full_id[4][0])]
				d_atom = full_id

				#Add or Update Edge
				if(g.has_edge(s_atom,d_atom)):
					g[s_atom][d_atom]['type'] |= typeCode[contact[2]]
				else:
					g.add_edge(s_atom,d_atom,type=typeCode[contact[2]],distance=contact[5])
								
				node = g.nodes[s_atom]
				node["type"] = node.get('type',0) | typeCode[contact[3]]
				node.update(s_data)
				
				node = g.nodes[d_atom]
				node.update(d_data)
				node["type"] = node.get('type',0) | typeCode[contact[4]]

			###########Compute Components
			number_comp = nx.number_connected_components(g)
			serial = None
			
			for comp in nx.connected_components(g):
				g_temp = g.subgraph(comp)
				g_temp = nx.Graph(g_temp)				
				g_temp = nx.convert_node_labels_to_integers(g_temp)
				
				g_temp.graph['id'] = g_id
				g_id+=1

				g_temp.graph['pdbid'] = pdbid

				g_temp.graph['source'],g_temp.graph['target'] = chains

				temp.append(g_temp)
						
	return sorted(temp,key=lambda g: g.graph['id'])

def fill_label_set(label_set):
	max_label = max(label_set)
	i = 1

	new_labels = set(label_set)
	while i < max_label:
		new_labels.add(i)
		i*=2
	return sorted(list(new_labels))
			

In [ ]:
import networkx as nx
graphs, node_labels, edge_labels = gen_graphs(contacts,typeCode,path=path)

In [ ]:
print(graphs)

In [ ]:
print(node_labels)

In [ ]:
print(edge_labels)

In [ ]:
##################### Clustering Analysis Start Here #############################
def genCountMatrix(graphs,node_labels,edge_labels,typeCode,matrix_type="full",method="simple",join_labels=True,filename="count_matrix.csv",path=""):
	
	node_labels = {k:v for v,k in enumerate(node_labels)}
	matrix_label_set = set()
	count_array = np.empty([len(graphs)],dtype=object)
	count_graph = 0

	if method == 'path':
		for g in graphs:
			shortPaths = [i for i in nx.all_pairs_shortest_path_length(g)]
			shortPaths = {i[0]:i[1] for i in shortPaths}

			temp = dict()
			for i in g.nodes:
				
				for j in g.nodes:
					if i >= j:
						continue
					
					labels = [g.nodes[i]["type"],g.nodes[j]["type"]]
					shortest_length = shortPaths[i][j]

					if join_labels:
						labels = [[l] for l in labels]
					else:
						labels = [mask.code(l) for l in labels]
					
					for l1 in labels[0]:
						for l2 in labels[1]:
							temp_labels = sorted(map(lambda x: node_labels[x],[l1,l2]))
							
							label = "%d-%d-%d"%(temp_labels[0],temp_labels[1],shortest_length)
							matrix_label_set.add(label)

							if matrix_type == "full":
								temp[label] = temp.get(label,0) + 1
							elif matrix_type == "bin":
								temp[label] = temp.get(label,1)

			count_array[count_graph] = temp
			count_graph += 1
	else:
		for g in graphs:
			temp = dict()
			for e in g.edges():
				labels = [g.nodes[e[0]]["type"],g.nodes[e[1]]["type"]]
				
				#### Added after join labels 
				if join_labels:
					labels = [[l] for l in labels]
				else:
					labels = [mask.code(l) for l in labels]
				
				for l1 in labels[0]:
					for l2 in labels[1]:
						temp_labels = sorted(map(lambda x: node_labels[x],[l1,l2]))
					
						label = "%d-%d"%(temp_labels[0],temp_labels[1])
						matrix_label_set.add(label)

						if matrix_type == "full":
							temp[label] = temp.get(label,0) + 1
						elif matrix_type == "bin":
							temp[label] = temp.get(label,1)

			count_array[count_graph] = temp
			count_graph += 1

	matrix_label_set = [i for i in matrix_label_set]
	matrix_label_set.sort()
	
	count_matrix = np.zeros([len(graphs),len(matrix_label_set)],dtype=int)

	for i in range(len(count_array)):
		for j in range(len(matrix_label_set)):			
			count_matrix[i][j] = count_array[i].get(matrix_label_set[j],0)
	
	ct_mtx_path = Path(path)/filename
	np.savetxt(ct_mtx_path,count_matrix,fmt='%d', delimiter=",")

	return count_matrix	




In [ ]:
	count_matrix_filename = "count_matrix.csv"
	if not (Path(path)/count_matrix_filename).exists():
		logging.info("---Generate counting matrix---")

		count_matrix = genCountMatrix(
					graphs,node_labels,edge_labels,typeCode,
					filename=count_matrix_filename,path=path)

	else:
		logging.info("---Load counting matrix---")
		count_matrix = np.genfromtxt((Path(path)/count_matrix_filename), delimiter=',')

In [ ]:
print(count_matrix)

In [ ]:
class SVD: 

	def __init__(self,count_matrix,scaler='std',with_std=False):
		self.scaler = scaler
		self.with_std = with_std
		self.runSVD(count_matrix)

	def runSVD(self,matrix):
		if self.scaler == "mabs":
			scaler_obj = MaxAbsScaler()
			X = scaler_obj.fit_transform(matrix)
		elif self.scaler == "std":
			scaler_obj = StandardScaler(with_std=self.with_std).fit(matrix)
			X = scaler_obj.fit_transform(matrix)

		self.U, self.s , self.v = np.linalg.svd(X,full_matrices=True)

	def getReduced(self,n_components=0):
		U = self.U
		s = self.s
		if n_components == 0:
			n_components = s.shape[0]
		matrix = -np.dot(U[:,:n_components],np.diag(s[:n_components]))

		return matrix

	def saveReduced(self,n_components=0,file='foo.csv'):
		np.savetxt(file, self.getReduced(n_components), fmt='%f',delimiter=",") 

	def getVarianceComposition(self,n=0):
		s = self.s
		var_comp = [0]
		for i in range(s.shape[0]):
			var_comp.append(var_comp[i]+s[i]*s[i])

		var_comp = [i for i in map(lambda x: x/var_comp[-1],var_comp)]

		if n == 0:
			return var_comp
		if 0 < n <= s.shape[0]:
			return var_comp[n]
		return 0

    
def find_n_clusters(count_matrix):
	n_counter = dict()
	max_gap_connected = dict()
	smallest_neighbors = count_matrix.shape[0]
	svd = SVD(count_matrix,scaler='std',with_std=False)
	var_comp = svd.getVarianceComposition()
	
	idx = 0
	while var_comp[idx] < 0.95:
		idx += 1

	cl_range = [i for i in range(idx,count_matrix.shape[1]+1)]

	for n_components in cl_range:				
		data_matrix = svd.getReduced(n_components)

		for factor in np.arange(0.01,0.91,0.01):
			n_neighbors = int(factor*data_matrix.shape[0])
			n_cluster, connected = eg.calculate_gap(data_matrix,
											n_neighbors=n_neighbors)
			if connected:
				max_gap_connected[n_components] = (n_cluster,n_neighbors)
				if n_neighbors < smallest_neighbors:
					smallest_neighbors = n_neighbors
					n_counter = dict()

				if n_neighbors == smallest_neighbors:
					n_counter[n_cluster] = n_counter.get(n_cluster,0) + 1

				break

	n_clusters = max(n_counter.items(),key= lambda x: x[1])[0]			
	n_cluster_candidates = list(filter(lambda x: x[1][1] == smallest_neighbors and x[1][0] == n_clusters,max_gap_connected.items()))
	i = int((len(n_cluster_candidates)-0.25)/2)
	params = n_cluster_candidates[i]

	#Returns n_components, n_clusters, k_neighbors
	return params[0],params[1][0],params[1][1], svd.getReduced(params[0])

In [ ]:
def spectral(data_matrix,n_clusters,n_neighbors=10):


	X_r = data_matrix
	algorithm = 'spectral'

	spectral = cl.SpectralClustering(n_clusters=n_clusters,
		affinity='nearest_neighbors',n_neighbors=n_neighbors)
	spectral.fit(X_r)
	
	y_pred = spectral.labels_.astype(np.int)
	return y_pred

In [ ]:
def read_clusters(clusters,graphs,path=""):
	if isinstance(clusters,str):
		try:
			file = (Path(path)/clusters).open()
			clusters = np.loadtxt(file,dtype=int,delimiter=",")
		except IOError as e:
			raise e
	#else:
	
	#clusters = []
	graphs_dict = dict()

	#Construct dictionary structure
	for key,graph in zip(clusters,graphs):
		graphs_dict[key] = graphs_dict.get(key,[]) + [graph]

	return graphs_dict



In [ ]:
	from pathlib import Path 
	from sklearn.preprocessing import StandardScaler
	import eigen_gap as eg
	from sklearn import cluster as cl
##################### Clustering #############################
	if not (Path(path)/"clusters.csv").exists():
		
		### Run SVD
		logging.info("---Run SVD on matrix---")

		n_components, n_clusters, k_neighbors, data_matrix = find_n_clusters(count_matrix)
				 
		logging.info("---Run Clustering---")
		res_cluster = spectral(data_matrix,n_clusters,k_neighbors)
		
		clusters_file_name = "clusters.csv"
		
		with (Path(path)/clusters_file_name).open(mode="w") as clusters_file:
			np.savetxt(clusters_file,res_cluster,fmt='%i', delimiter=",")
		
		clusters = read_clusters(res_cluster,graphs)
	else:
		logging.info("---Load Clusters---")
		clusters_file_name = "clusters.csv"
		clusters = read_clusters(clusters_file_name,graphs,path=path)
        



In [ ]:
print(n_components)

In [ ]:
print(n_clusters)

In [ ]:
print(k_neighbors)

In [ ]:
print(data_matrix)

In [ ]:
print(clusters)

In [ ]:
##################### Conserved SubStructure Mining Start Here #############################


##FSM for each cluster
def gen_gSpan_entries(graphs,clusters,supports,node_labels,edge_labels,type_code,path='',gSpan_path=''):
	
	gSpanFName = 'entry.gsp'

	with cd(path):

		for key,graph_list in sorted(clusters.items(),
										key=lambda x: x[0]):
			gSpanFName='%s.gsp'%key
			multigraph_to_gspan(graph_list,
					node_labels,edge_labels,type_code,gspan_fname=gSpanFName)

            
def multigraph_to_gspan(graphs,node_labels,edge_labels,mask,gspan_fname="entry.gsp"):
	# print("Debug multigraph_to_gspan")
	# print('Edge labels')
	# print(edge_labels)
	edge_labels = fill_label_set(edge_labels)
	# print('New edge labels')
	# print(edge_labels)
	node_labels = fill_label_set(node_labels)
	graphs_holder = graphs
	graphs = []
	node_map = None
	node_labels = {k:v for v,k in enumerate(node_labels)}
	edge_labels = {k:v for v,k in enumerate(edge_labels)}
	
	for i in range(len(graphs_holder)):
		
		node_map = {k:v for v,k in enumerate(sorted(graphs_holder[i].nodes()))}
		graphs.append(copy.deepcopy(graphs_holder[i]))
		graphs[i].graph["node_map"] = node_map
	
		nx.relabel_nodes(graphs[i],node_map,copy=False)

	temp_str = ""

	for i in range(len(graphs)):

		temp_str1 = ["t # " + str(i) + "\n"]
		nodes = sorted(graphs[i].nodes)

		for j in nodes:
			if node_labels:
				node_type = graphs[i].nodes[j]['type']
				temp_str1 += ["v %d %d\n" % (j,node_labels[node_type])]
			else:
				temp_str1 += ["v %d\n" % (j)]
			
		for j in graphs[i].edges(data=True):
			ml = len(mask.code(j[2]["type"]))
			for l in mask.code(j[2]["type"]):
				#print(j[0],j[1],l)
				temp_str1 += ["e %d %d %d\n" % (j[0],j[1],edge_labels[l])]
				
		temp_str += "".join(temp_str1)
		count = 0
		
	with open(gspan_fname,"w") as gspan_file:
		gspan_file.write(temp_str)

	node_map = [{k:v for v,k in enumerate(sorted(g.nodes()))} for g in graphs_holder]
	return node_map		


def fill_label_set(label_set):
	max_label = max(label_set)
	i = 1

	new_labels = set(label_set)
	while i < max_label:
		new_labels.add(i)
		i*=2
	return sorted(list(new_labels))
			
            
                       
def runGSpan(graphs,clusters,supports,node_labels,edge_labels,path='',gSpan_path=''):
	
	
	graphs_dict = clusters
	
	gSpan_out = (Path(path)/'gSpan.txt').open(mode='w') #gSpan log
	gSpan_results = dict()
	
	#Change context 
	with cd(path):

		for key,graph_list in sorted(graphs_dict.items(), key=lambda x: x[0]):
			
			gSpanFName = str(key)
			
			temp_gSpan_results = []
			for min_sup in supports:
				shutil.copyfile(gSpanFName + '.gsp', gSpanFName + '_.gsp')
				call([gSpan_path,"-f",gSpanFName + '_.gsp',"-s",str(min_sup),"-o","-i"],stdout=gSpan_out, shell=True)
				Path(gSpanFName + '_.gsp').rename('%s_%s.fp'%(key,min_sup))

				temp_gSpan_results.append('%s_%s.fp'%(key,min_sup))

			gSpan_results[int(key)] = temp_gSpan_results

		gSpan_results = {'results':gSpan_results, 'supports':supports}
		with open('gSpan.fp','w') as out_gspan_files:
			json.dump(gSpan_results,out_gspan_files,indent=4)

	return gSpan_results,graphs_dict


def gspan_to_graph(filename,node_labels,edge_labels,multigraph=False):

	with open(filename,"r") as file:
		patterns = [ [line for line in group.split("\n")] for group in file.read().split("\n\n")]
	graphs = []
	
	node_labels = {k:v for k,v in enumerate(node_labels)}
	edge_labels = {k:v for k,v in enumerate(edge_labels)}


	for i in patterns:
		#print(i)        
		if len(i) < 2:
			continue 
		i[0] = i[0].split()
		#print(i[0])
		temp = None
		if(multigraph):
			temp = nx.MultiGraph()
		else:
			temp = nx.Graph()
            
		for j in i[1:]:
			
			#print (j) 
			j = j.split()
			#if j[0] != "":
			#print (j)             
			#print ("v") 
			if j:    # Addition of if j:
				#print (j) 
				if j[0] == "v":
					#print (j[0] +"=="+ "v")
					temp.add_node(int(j[1]),type=node_labels[int(j[2])])
				if j[0] == "e":
					#print (j[0] +"=="+ "e")
					if(temp.has_edge(int(j[1]),int(j[2]))):
						temp[int(j[1])][int(j[2])]['type'] += edge_labels[int(j[3])]
					else:
						temp.add_edge(int(j[1]),int(j[2]),type=edge_labels[int(j[3])])
				if j[0] == "x":
					temp.graph["ocur"] = j[1:]
		#print(i[0],i[2])
		#print(i[4])
		temp.graph['support'] = i[0],i[4]
		temp.graph['id'] = i[0],i[2]
		
		graphs.append(temp)
	
	return graphs
 
    
def read_gSpan_results(node_labels,edge_labels,filename="gSpan.fp",path=""):

	with (Path(path)/filename).open() as filename_list:
		result_files = json.load(filename_list)

	gSpan_results = dict()
	supports = result_files['supports']
	with cd(path):		
		for cluster,files in result_files['results'].items():
			temp_results = dict()
			for filename in files:
				key = filename.split('_')[1].split('f')[0][:-1]
				temp_results[key] = gspan_to_graph(filename,node_labels,edge_labels)
			
			gSpan_results[int(cluster)] = temp_results
	return gSpan_results,supports


class cd:
    """Context manager for changing the current working directory"""
    def __init__(self, newPath):
        self.newPath = Path(newPath).expanduser()

    def __enter__(self):
        self.savedPath = Path.cwd()
        os.chdir(str(self.newPath))

    def __exit__(self, etype, value, traceback):
        os.chdir(str(self.savedPath))
        



In [ ]:
	##################### Run gSpan #############################
	import copy, shutil
	import os
	from subprocess import call


    
	if not (Path(path)/"gSpan.fp").exists():

		logging.info("---Run gSpan---")
		supports = [ "%.1f"%i for i in np.arange(0.5,1.09,0.1)]
		gen_gSpan_entries(graphs,clusters,supports,
								node_labels,edge_labels,typeCode,path=path,gSpan_path=GSPANPATH)

		graph_results,clusters = runGSpan(graphs,clusters,supports,
								node_labels,edge_labels,path=path,gSpan_path=GSPANPATH)
		#}'''
	
	
	logging.info("---Read gSpan results---")
	graph_results,supports = read_gSpan_results(node_labels,edge_labels,filename="gSpan.fp",path=path)


In [ ]:
print(graph_results)

In [ ]:
print(supports)

In [ ]:
#Subgraph Isomorphism Start Here
def getMaximalGraphs(clusters,file="maximal.json",path=""):	

	j_maximal = []
	maximal_graphs = []
	
	for n_cluster,cluster in clusters.items():
	
		j_temp = dict()
		temp = dict()
	
		for min_sup, graphs in sorted(cluster.items()):
			#print(min_sup)
			#print(graphs)
			g =	maximal(graphs)
			#print(g)
			temp[min_sup] = [ {"graph": j["graph"], "l_graph": j["l_graph"]} for j in g ]
			print(temp[min_sup]) 
			j_temp[min_sup] = ["".join([i for i in nx.generate_gml(j["graph"])]) for j in g]  # here

		maximal_graphs.append(temp)
		j_maximal.append(j_temp)
	

	with (Path(path) / file ).open(mode='w') as j_patterns_file:
		j_patterns_file.write(json.dumps(j_maximal,indent=4))

	return maximal_graphs

In [ ]:
def maximal(graphs):

	############# Filter: remove 1-vertex graphs
	graphs = [i for i in filter(lambda x: x.number_of_nodes() > 1,graphs)]

	if not graphs:
		return []

	graphs.sort(key=lambda x: -x.number_of_nodes())

	### Generate Line Graphs
	graphs = [{"graph": g, "l_graph" : line_graph(g)} for g in graphs]
	
	############# Get maximals

	######## Node Split
	graphs_holder = graphs
	graphs = []
	last_num_nodes =-1
	while graphs_holder:
		num_nodes = graphs_holder[-1]["graph"].number_of_nodes()
		if  num_nodes != last_num_nodes:
			graphs.append([])
			last_num_nodes = num_nodes

		graphs[-1].append(graphs_holder.pop())

	######## Edge Split	
	graphs = [sorted(i,key = lambda x: -x['graph'].number_of_edges()) for i in graphs]

	######## Graph Subgraph Isomorphism (Level 1)
	nm = lambda x,y: x['type'] >= y['type'] and (x['type'] & y['type'])
	em = nx.isomorphism.numerical_node_match(["type"],[""])

	marked = [np.full((len(g)),False) for g in graphs]
	
	for g in range(1,len(graphs)):
		count = 0

		for i in range(len(graphs[g])):
			j_size = len(graphs[g-1])
			
			if count >= j_size:
				break

			for j in range(j_size):
				if marked[g-1][j]:
					continue

				m = nx.isomorphism.GraphMatcher(graphs[g][i]["l_graph"],graphs[g-1][j]["l_graph"],
							edge_match=em,node_match=nm)

				if m.subgraph_is_isomorphic():
					count+=1		
					marked[g-1][j] = True

				if count >= j_size:
					break
			

	graphs_holder = [np.array(g) for g in graphs]
	graphs = [g[i] for g,i in zip(graphs_holder,np.invert(np.array(marked)))]
	graphs = np.concatenate(np.flip(graphs))
	
	######## Graph Subgraph Isomorphism (Level 2)
	nm = lambda x,y: x['type'] >= y['type'] and (x['type'] & y['type'])
	em = nx.isomorphism.numerical_node_match(["type"],[""])
	
	marked = np.full((len(graphs)),False)
	count = 0
	
	for i in range(len(graphs)):
	
		for j in range(len(graphs)):
			if(i == j or marked[j]):
				continue

			if(graphs[i]["graph"].number_of_nodes() >= graphs[j]["graph"].number_of_nodes()):
				
				m = nx.isomorphism.GraphMatcher(graphs[i]["l_graph"],graphs[j]["l_graph"],
							edge_match=em,node_match=nm)

				if(m.subgraph_is_isomorphic()):
					count+=1	
					marked[j] = True

	for i,g in enumerate(graphs):
		g["graph"].graph['id'] = i

	return list(graphs[np.invert(marked)])

In [ ]:
def line_graph(g):

	#### Create line_graph
	gn = nx.Graph(nx.line_graph(g))

	#### Labeling Edges
	for n1,n2 in gn.edges():
		if n1[0] == n2[0]:
			gn[n1][n2]["type"] = g.nodes[n1[0]]["type"]
		elif n1[0] == n2[1]:
			gn[n1][n2]["type"] = g.nodes[n1[0]]["type"]
		else:
			gn[n1][n2]["type"] = g.nodes[n1[1]]["type"]
		gn[n1][n2]["type"] = 0

	#### Labeling Nodes
	for n in gn.nodes():
		gn.nodes[n]['type'] = g[n[0]][n[1]]['type']

	return gn

In [ ]:
############## Maximal

if not (Path(path)/"maximal.json").exists():
	logging.info("---Get maximal graphs---")
	maximal_patterns = getMaximalGraphs(graph_results,path=path)